# Dazo v0 — ProofWriter recurrent-depth pilot

This notebook is safe to rerun in the same Colab runtime. It updates `8dazo/dazo`, reuses older nested ProofWriter data/checkpoints, trains only when no completed pilot exists, and evaluates the same checkpoint at 1/2/4/6/8 recurrent steps.

**Runtime:** GPU. Tesla T4 is enough for the pilot.


## 0. Setup / update repo safely


In [ ]:
!nvidia-smi || true

from pathlib import Path
import os, shutil, subprocess

os.chdir('/content')
repo = Path('/content/dazo')

if (repo / '.git').exists():
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo), 'reset', '--hard', 'origin/main'], check=True)
elif repo.exists():
    stale = Path('/content/dazo-stale')
    if stale.exists():
        shutil.rmtree(stale)
    repo.rename(stale)
    subprocess.run(['git', 'clone', '-q', 'https://github.com/8dazo/dazo.git', str(repo)], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/8dazo/dazo.git', str(repo)], check=True)

os.chdir(repo)
print('repo:', Path.cwd())
print('commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode().strip())
!pip -q install -e ".[train]"


## 1. Architecture + loss invariants


In [ ]:
!pytest -q tests/test_core.py tests/test_losses.py


## 2. Prepare or recover ProofWriter pilot

This cell first searches the runtime for a previous nested `proofwriter-pilot` dataset. If it finds one, it copies the three JSONL files into the current repo. Otherwise it streams a fresh 3k/1k/1k pilot.


In [ ]:
from pathlib import Path
import shutil, subprocess, os

os.chdir('/content/dazo')
target = Path('/content/dazo/data/proofwriter-pilot')
target.mkdir(parents=True, exist_ok=True)
names = ['train.jsonl', 'validation.jsonl', 'test.jsonl']

if not all((target / n).exists() for n in names):
    old_tests = [
        p for p in Path('/content').glob('**/data/proofwriter-pilot/test.jsonl')
        if p.resolve() != (target / 'test.jsonl').resolve()
    ]
    recovered = False
    for test_file in sorted(old_tests, key=lambda p: p.stat().st_mtime, reverse=True):
        src = test_file.parent
        if all((src / n).exists() for n in names):
            print('Recovering ProofWriter data from:', src)
            for n in names:
                shutil.copy2(src / n, target / n)
            recovered = True
            break
    if not recovered:
        subprocess.run([
            'python', 'scripts/prepare_proofwriter.py',
            '--output', str(target),
            '--train-max-depth', '3',
            '--limit-train', '3000',
            '--limit-eval', '1000',
        ], check=True)

for n in names:
    p = target / n
    print(n, p.exists(), p)


## 3. Train the recurrent head only if needed

If a previous completed pilot checkpoint exists anywhere under `/content`, this cell skips training automatically. On a fresh runtime it trains one epoch on the pilot.


In [ ]:
from pathlib import Path
import os, subprocess

os.chdir('/content/dazo')
existing_ckpts = list(Path('/content').glob('**/outputs/dazo-proofwriter-pilot/final/config.json'))

if existing_ckpts:
    newest = max((p.parent for p in existing_ckpts), key=lambda p: p.stat().st_mtime)
    print('Completed pilot checkpoint already exists; skipping training:')
    print(newest)
else:
    cmd = [
        'python', 'train.py',
        '--config', 'configs/dazo-v0-small.json',
        '--train', 'data/proofwriter-pilot/train.jsonl',
        '--eval', 'data/proofwriter-pilot/validation.jsonl',
        '--output', 'outputs/dazo-proofwriter-pilot',
        '--epochs', '1',
        '--batch-size', '4',
        '--grad-accum', '2',
        '--lr', '2e-4',
        '--depth-budgets', '1,2,3,4,6,8',
    ]
    subprocess.run(cmd, check=True)


## 4. Recover checkpoint + test-time compute scaling

This is the main Gate-1 experiment. It discovers the newest completed checkpoint and the current absolute test-data path, then evaluates with the latest source code. If evaluation fails, this cell prints the full child-process stdout and stderr.


In [ ]:
from pathlib import Path
import os, subprocess

repo = Path('/content/dazo')
os.chdir(repo)

candidates = list(Path('/content').glob('**/outputs/dazo-proofwriter-pilot/final/config.json'))
if not candidates:
    raise RuntimeError('No completed Dazo pilot checkpoint found. Run section 3 first.')

checkpoint = max((p.parent for p in candidates), key=lambda p: p.stat().st_mtime)
test_data = repo / 'data/proofwriter-pilot/test.jsonl'
if not test_data.exists():
    data_candidates = list(Path('/content').glob('**/data/proofwriter-pilot/test.jsonl'))
    if not data_candidates:
        raise RuntimeError('No ProofWriter pilot test.jsonl found. Run section 2 first.')
    test_data = max(data_candidates, key=lambda p: p.stat().st_mtime)

report = repo / 'outputs/dazo-proofwriter-pilot/gate1-metrics.json'
report.parent.mkdir(parents=True, exist_ok=True)

print('Using checkpoint:', checkpoint)
print('Using test data :', test_data)
print('Using evaluator :', repo / 'evaluate.py')

cmd = [
    'python', str(repo / 'evaluate.py'),
    '--model', str(checkpoint),
    '--data', str(test_data),
    '--batch-size', '8',
    '--loops', '1,2,4,6,8',
    '--output', str(report),
]
result = subprocess.run(cmd, cwd=str(repo), text=True, capture_output=True)

print(result.stdout)
if result.stderr:
    print('--- stderr ---')
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f'Dazo evaluation failed with exit code {result.returncode}. Full stderr is printed above.')

print('Saved:', report)


## 5. Gate-1 summary


In [ ]:
import json
from pathlib import Path

report = Path('/content/dazo/outputs/dazo-proofwriter-pilot/gate1-metrics.json')
r = json.loads(report.read_text())

for loop, m in r['loops'].items():
    print(
        f"loops={loop:>2} "
        f"accuracy={m['accuracy']:.4f} "
        f"brier={m['brier']:.4f} "
        f"ece={m['ece']:.4f} "
        f"by_depth={m['by_depth']}"
    )

print('overthinking_rate =', r.get('overthinking_rate'))
print('ever_correct_then_final_wrong =', r.get('ever_correct_then_final_wrong'))
print('transitions =')
print(json.dumps(r.get('correctness_transitions', {}), indent=2))


## 6. What counts as a useful Gate-1 signal?

The 3k/1-epoch pilot is only a pipeline and signal check. We want deeper ProofWriter depths to benefit from more recurrent loops without large shallow-depth regressions. If the recurrence curve is flat or harmful, we revise Dazo-0 before scaling. If it is meaningfully positive, the next run should use 20k–50k shallow-depth examples and 2–3 epochs.
